# Tech Challenge Fase 2 - Classificação da Qualidade de Vinhos

## 02. Pré-processamento

Segundo notebook do projeto. Recebe a base gerada no `01_eda.ipynb` (já sem duplicados e com a coluna `high_quality` criada) e prepara os dados para modelagem: confere nulos, decide a estratégia de normalização e cria as variáveis derivadas de feature engineering.

## 1. Instalação e importação das bibliotecas

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

os.makedirs("data/processed", exist_ok=True)

## 2. Carregar a base gerada no notebook anterior

In [2]:
eda_path = "data/processed/wine_quality_eda.csv"

if not os.path.exists(eda_path):
    try:
        from google.colab import files
        print("Faça upload do arquivo wine_quality_eda.csv gerado no notebook 01_eda")
        uploaded = files.upload()
        uploaded_name = list(uploaded.keys())[0]
        os.replace(uploaded_name, eda_path)
    except Exception:
        raise FileNotFoundError("Arquivo wine_quality_eda.csv não encontrado. Rode o notebook 01_eda primeiro.")

df_model = pd.read_csv(eda_path)
print("Base carregada!")
print("Linhas:", df_model.shape[0])
print("Colunas:", df_model.shape[1])
df_model.head()

Base carregada!
Linhas: 1018
Colunas: 14


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id,high_quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3,0
4,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5,5,0


## 3. Nulos

Já foi conferido no notebook anterior que a base não tem valores nulos. Confirmando de novo aqui, já que este notebook é o responsável formal pelo tratamento de dados faltantes.

In [3]:
missing = df_model.isna().sum().sort_values(ascending=False)
total_nulos = missing.sum()
print("Total de valores nulos na base:", total_nulos)
missing[missing > 0]

Total de valores nulos na base: 0


Series([], dtype: int64)

Como não há nulos, nenhum tratamento de imputação é necessário nesta base. Se houvesse, a estratégia dependeria da variável: para variáveis físico-químicas contínuas como estas, o mais indicado seria imputação pela mediana (menos sensível a outliers que a média), aplicada separadamente em treino e teste para não vazar informação do teste para o treino.

## 4. Normalização e padronização

Nem todo modelo precisa de escala igual entre as variáveis. Random Forest e Gradient Boosting, por serem baseados em divisões de árvore, são invariantes à escala das variáveis, então não é necessário padronizar nada para eles. A Regressão Logística, por outro lado, é sensível à escala: variáveis com valores muito maiores que as outras (como `total sulfur dioxide`, que chega a dezenas, contra `chlorides`, que fica na casa dos centésimos) dominariam o cálculo do gradiente e distorceriam os coeficientes.

A decisão aqui foi aplicar `StandardScaler` (média 0, desvio padrão 1) apenas dentro do pipeline da Regressão Logística, no notebook de modelagem, e não escalar a base inteira agora. Isso evita um erro comum: se a padronização fosse feita antes da separação entre treino e teste, a média e o desvio padrão calculados incluiriam dados do teste, causando vazamento de informação. Ao colocar o `StandardScaler` dentro do pipeline, ele é ajustado (`fit`) só com os dados de treino em cada divisão.

## 5. Feature engineering

Antes de seguir para a modelagem, vale criar algumas variáveis derivadas que resumem combinações de indicadores físico-químicos já usadas na prática enológica. A ideia não é substituir as variáveis originais, e sim dar ao modelo combinações que já fazem sentido tecnicamente:

- `acidez_total`: soma da acidez fixa com a acidez volátil, uma leitura mais completa da acidez total do vinho do que olhar cada uma separadamente.
- `razao_so2_livre`: proporção do dióxido de enxofre livre em relação ao total. É um indicador usado na prática para saber quanto SO2 realmente está disponível para proteger o vinho, já que o SO2 combinado já reagiu e perde parte do efeito conservante.
- `alcool_por_densidade`: razão entre teor alcoólico e densidade, tentando capturar em uma única variável a relação inversa entre as duas que já apareceu na correlação do notebook anterior.

In [4]:
df_model["acidez_total"] = df_model["fixed acidity"] + df_model["volatile acidity"]
df_model["razao_so2_livre"] = df_model["free sulfur dioxide"] / df_model["total sulfur dioxide"].replace(0, np.nan)
df_model["razao_so2_livre"] = df_model["razao_so2_livre"].fillna(0)
df_model["alcool_por_densidade"] = df_model["alcohol"] / df_model["density"]

id_cols = ["Id"] if "Id" in df_model.columns else []
target_cols = ["quality", "high_quality"]
feature_cols = [c for c in df_model.columns if c not in id_cols + target_cols]

print("Total de features após feature engineering:", len(feature_cols))
feature_cols

Total de features após feature engineering: 14


['fixed acidity',
 'volatile acidity',
 'citric acid',
 'residual sugar',
 'chlorides',
 'free sulfur dioxide',
 'total sulfur dioxide',
 'density',
 'pH',
 'sulphates',
 'alcohol',
 'acidez_total',
 'razao_so2_livre',
 'alcool_por_densidade']

## 6. Salvar base processada para a próxima etapa

In [5]:
processed_path = "data/processed/wine_quality_features.csv"
df_model.to_csv(processed_path, index=False)
print("Base salva em:", processed_path)
print("Linhas:", len(df_model), "| Colunas:", df_model.shape[1])

try:
    from google.colab import files
    files.download(processed_path)
except Exception:
    pass

Base salva em: data/processed/wine_quality_features.csv
Linhas: 1018 | Colunas: 17


O arquivo `wine_quality_features.csv` baixado aqui é a entrada do próximo notebook (`03_modelagem.ipynb`). Ele já tem as três variáveis derivadas e está pronto para a separação entre treino e teste.